# Task 2: Custom BPE Tokenizer and Tiny Causal LM



In [1]:
from collections import Counter
import re
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)


In [2]:
class SimpleBPE:
    def __init__(self, num_merges=20):
        self.num_merges = num_merges
        self.merges = []
        self.vocab = {}

    def _word_to_symbols(self, word):
        # We add </w> to mark the end of a word.
        return tuple(list(word) + ["</w>"])

    def _get_pair_counts(self, words):
        pair_counts = Counter()
        for tokens, freq in words.items():
            for i in range(len(tokens) - 1):
                pair_counts[(tokens[i], tokens[i + 1])] += freq
        return pair_counts

    def _merge_pair(self, pair, words):
        merged_words = {}
        bigram = " ".join(pair)
        replacement = "".join(pair)

        for tokens, freq in words.items():
            token_string = " ".join(tokens)
            token_string = token_string.replace(bigram, replacement)
            merged_words[tuple(token_string.split())] = freq
        return merged_words

    def fit(self, text):
        words = re.findall(r"[a-zA-Z']+", text.lower())
        word_counts = Counter(words)
        encoded_words = {self._word_to_symbols(word): freq for word, freq in word_counts.items()}

        for _ in range(self.num_merges):
            pair_counts = self._get_pair_counts(encoded_words)
            if not pair_counts:
                break

            best_pair = pair_counts.most_common(1)[0][0]
            self.merges.append(best_pair)
            encoded_words = self._merge_pair(best_pair, encoded_words)

        vocab_tokens = set()
        for tokens in encoded_words:
            vocab_tokens.update(tokens)

        self.vocab = {token: idx for idx, token in enumerate(sorted(vocab_tokens))}
        return self

    def encode_word(self, word):
        tokens = list(word) + ["</w>"]
        for pair in self.merges:
            i = 0
            while i < len(tokens) - 1:
                if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                    tokens[i : i + 2] = ["".join(pair)]
                else:
                    i += 1
        return tokens

    def encode(self, text):
        words = re.findall(r"[a-zA-Z']+", text.lower())
        ids = []
        for word in words:
            pieces = self.encode_word(word)
            for piece in pieces:
                if piece in self.vocab:
                    ids.append(self.vocab[piece])
        return ids


In [3]:
sample_text = """large language models learn from many text examples.
language models generate the next token one step at a time.
bpe tokenization helps build useful subword units.
"""

bpe = SimpleBPE(num_merges=25).fit(sample_text)
token_ids = bpe.encode(sample_text)

print("Vocabulary size:", len(bpe.vocab))
print("First 30 token ids:", token_ids[:30])


Vocabulary size: 37
First 30 token ids: [20, 15, 19, 23, 21, 1, 28, 25, 13, 28, 26, 22, 0, 22, 1, 24, 35, 0, 31, 12, 11, 1, 22, 27, 21, 30, 19, 23, 14, 9]


In [4]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-1, -2)
        scores = scores / (q.size(-1) ** 0.5)

        seq_len = input_ids.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=input_ids.device))
        scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=-1)
        context = weights @ v

        logits = self.ffn(context)
        return logits


In [5]:
# We create small training examples from our token sequence.
sequence_length = 6
examples_x = []
examples_y = []

for i in range(len(token_ids) - sequence_length):
    x = token_ids[i : i + sequence_length]
    y = token_ids[i + 1 : i + sequence_length + 1]
    examples_x.append(x)
    examples_y.append(y)

X = torch.tensor(examples_x, dtype=torch.long)
Y = torch.tensor(examples_y, dtype=torch.long)

model = TinyCausalLM(vocab_size=len(bpe.vocab), embed_dim=32, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(30):
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")


Epoch 10, Loss: 2.2747
Epoch 20, Loss: 1.1388
Epoch 30, Loss: 0.7968
